# Dual-Strait Portfolio Optimization
## Strait of Hormuz + Bab al-Mandeb: Second-Order Commodity Plays

**Methodology:** Same two-level hierarchical Monte Carlo as the Hormuz-only portfolio, extended to 6 buckets by adding Container Shipping (the BaM-specific exposure).

**Key design changes vs Hormuz-only:**
- Container Shipping bucket added (GSL — shortest charter repricing window)
- Tankers cap raised from 20% → 30% (dual closure creates extreme ton-mile spike)
- Other bucket ceilings trimmed to accommodate new bucket and higher tanker weight

**Optimization window:** 2022–2024 (same as Hormuz portfolio, for apples-to-apples comparison)  
**Validation / backtest:** 2025–present (fully out-of-sample)  
**Strategic intent:** Capture the pre-closure run-up in BaM-sensitive stocks, mirroring the pre-Feb-28-2026 gains the Hormuz portfolio captured before the strait officially closed.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Six-Bucket Portfolio Definition

Five buckets carried over from the Hormuz portfolio (same stocks, same within-bucket weights), plus Container Shipping as the BaM-specific addition.

In [ ]:
BUCKETS = {
    'Aluminum': {
        'AA':    0.50,   # Alcoa — hydro-powered; captures full revenue upside without full energy cost pass-through
        'CENX':  0.25,   # Century Aluminum — highest beta to Al price
        'RIO':   0.25,   # Rio Tinto — NYSE-listed; comparable hydro exposure (Iceland, Quebec)
    },
    'LNG_Gas': {
        'LNG':   0.25,   # Cheniere Energy — largest US LNG exporter
        'EQT':   0.25,   # EQT Corp — largest US natural gas producer
        'AR':    0.25,   # Antero Resources — high LNG export leverage
        'GLNG':  0.25,   # Golar LNG — shipping + FLNG assets; benefits from longer transit distances
    },
    'Fertilizer': {
        'CF':    1/3,    # CF Industries — largest US nitrogen producer; domestic gas pricing advantage
        'MOS':   1/3,    # Mosaic — US potash + phosphate
        'IPI':   1/3,    # Intrepid Potash — only US domestic potash pure-play; benefits from BOTH catalysts
    },
    'Defense': {
        'LMT':   0.60,   # Lockheed Martin — THAAD + PAC-3 MSE (the consumable that must be reordered)
        'RTX':   0.30,   # RTX Corp — SM-3, Patriot system; naval angle directly relevant to Gulf ops
        'NOC':   0.10,   # Northrop Grumman — IBCS battle network integration; long-cycle sticky revenue
    },
    'Tankers': {
        'FRO':   1/3,    # Frontline — VLCC crude tankers; most benefit from long-haul rerouting
        'STNG':  1/3,    # Scorpio Tankers — product tankers; different cargo for diversification
        'INSW':  1/3,    # International Seaways — mixed crude + product fleet
    },
    'Container': {
        'GSL':   1.00,   # Global Ship Lease — 3.3yr avg charter (vs CMRE's 5.9yr); fastest repricing into rate spike
        # MATX (Matson) is the US-flagged alternative if a second stock is added
    },
}

ALL_TICKERS = [t for bucket in BUCKETS.values() for t in bucket]
print(f"{len(ALL_TICKERS)} tickers across {len(BUCKETS)} buckets")
print(ALL_TICKERS)

## 2. Pull Price Data (2022–2024)

Same optimization window as the Hormuz portfolio — enables apples-to-apples comparison.

In [ ]:
START = '2022-01-01'
END   = '2024-12-31'

raw = yf.download(ALL_TICKERS, start=START, end=END, auto_adjust=True, progress=False)['Close']

print(f"Downloaded: {raw.shape[0]} trading days, {raw.shape[1]} tickers")
print(f"Date range: {raw.index[0].date()} → {raw.index[-1].date()}")
print("\nMissing data per ticker:")
print(raw.isnull().sum().to_string())

In [ ]:
returns = raw.pct_change().dropna(how='all')
returns = returns.ffill().dropna()

print(f"Returns shape after cleaning: {returns.shape}")
print(f"\nAnnualised return (mean × 252):")
ann_ret = (returns.mean() * 252).sort_values(ascending=False)
print(ann_ret.apply(lambda x: f"{x:.1%}").to_string())

## 3. Construct Bucket Return Series

In [ ]:
bucket_returns = pd.DataFrame(index=returns.index)

for bucket_name, weights in BUCKETS.items():
    tickers = list(weights.keys())
    w = np.array(list(weights.values()))
    w = w / w.sum()
    bucket_returns[bucket_name] = (returns[tickers] * w).sum(axis=1)

print("Bucket annualised stats (2022–2024):")
bucket_ann = bucket_returns.mean() * 252
bucket_vol = bucket_returns.std() * np.sqrt(252)
bucket_sr  = bucket_ann / bucket_vol

summary = pd.DataFrame({'Ann. Return': bucket_ann, 'Ann. Vol': bucket_vol, 'Sharpe (rf=0)': bucket_sr})
print(summary.applymap(lambda x: f"{x:.3f}").to_string())

print("\n6-Bucket correlation matrix:")
print(bucket_returns.corr().applymap(lambda x: f"{x:.2f}").to_string())

## 4. Monte Carlo Optimization — 6-Bucket Dual-Strait

Constraint changes vs Hormuz-only:

| Bucket | Hormuz min/max | Dual-Strait min/max | Rationale |
|---|---|---|---|
| Aluminum | 10% / 40% | 5% / 35% | Thesis unchanged; floor lowered — MC will size down if correlation is unfavorable |
| LNG_Gas | 10% / 40% | 10% / 40% | Thesis strengthened under dual closure; unchanged ceiling |
| Fertilizer | 10% / 40% | 5% / 30% | Both catalysts intact; smaller floor to give MC flexibility |
| Defense | 10% / 40% | 10% / 40% | Structurally insulated; unchanged ceiling |
| Tankers | 5% / 20% | **5% / 25%** | **Conservatively capped** — tankers are the most resolution-sensitive bucket; freight rates normalize within days if either strait reopens. Unconstrained MC wanted 34.5%; capped here to limit tail risk. |
| Container | — | 5% / 20% | BaM-specific; GSL is the vehicle; single stock so capped conservatively |

In [ ]:
RF_RATE = 0.043  # ~4.3% 3-month T-bill, late 2024

N_SIM        = 10_000
BUCKET_NAMES = list(BUCKETS.keys())
N_BUCKETS    = len(BUCKET_NAMES)

BUCKET_BOUNDS = {
    'Aluminum':   (0.05, 0.35),
    'LNG_Gas':    (0.10, 0.40),
    'Fertilizer': (0.05, 0.30),
    'Defense':    (0.10, 0.40),
    'Tankers':    (0.05, 0.25),  # conservative cap — most resolution-sensitive bucket
    'Container':  (0.05, 0.20),  # BaM-specific; capped conservatively (single stock)
}
W_MINS = np.array([BUCKET_BOUNDS[b][0] for b in BUCKET_NAMES])
W_MAXS = np.array([BUCKET_BOUNDS[b][1] for b in BUCKET_NAMES])

cov_matrix   = bucket_returns.cov() * 252
mean_returns = bucket_returns.mean() * 252

def sample_constrained_weights(n_target, w_mins, w_maxs, rng, batch=50_000):
    n_assets = len(w_mins)
    collected = []
    while sum(len(c) for c in collected) < n_target:
        raw     = rng.dirichlet(np.ones(n_assets), size=batch)
        clipped = np.clip(raw, w_mins, w_maxs)
        normed  = clipped / clipped.sum(axis=1, keepdims=True)
        valid   = np.all((normed >= w_mins) & (normed <= w_maxs), axis=1)
        collected.append(normed[valid])
    return np.vstack(collected)[:n_target]

weights_matrix = sample_constrained_weights(N_SIM, W_MINS, W_MAXS, rng)

port_returns = weights_matrix @ mean_returns.values
port_vols    = np.sqrt(
    np.einsum('ij,jk,ik->i', weights_matrix, cov_matrix.values, weights_matrix)
)
port_sharpes = (port_returns - RF_RATE) / port_vols

print(f"Simulated {N_SIM:,} portfolios")
print(f"Return range:  {port_returns.min():.1%} – {port_returns.max():.1%}")
print(f"Vol range:     {port_vols.min():.1%} – {port_vols.max():.1%}")
print(f"Sharpe range:  {port_sharpes.min():.3f} – {port_sharpes.max():.3f}")

## 5. Efficient Frontier

In [ ]:
best_idx   = np.argmax(port_sharpes)
minvol_idx = np.argmin(port_vols)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(port_vols, port_returns, c=port_sharpes, cmap='plasma', alpha=0.4, s=8, linewidths=0)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')

ax.scatter(port_vols[best_idx], port_returns[best_idx],
           marker='*', s=300, color='gold', edgecolors='black', linewidths=0.8, zorder=5,
           label=f'Max Sharpe  ({port_sharpes[best_idx]:.3f})')
ax.scatter(port_vols[minvol_idx], port_returns[minvol_idx],
           marker='D', s=120, color='cyan', edgecolors='black', linewidths=0.8, zorder=5,
           label=f'Min Vol  ({port_vols[minvol_idx]:.1%})')

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_xlabel('Annualised Volatility', fontsize=12)
ax.set_ylabel('Annualised Return', fontsize=12)
ax.set_title('Efficient Frontier — 6-Bucket Dual-Strait Monte Carlo (10,000 simulations)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('dual_strait_efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dual_strait_efficient_frontier.png")

## 6. Optimal Weights

In [ ]:
optimal_bucket_weights = dict(zip(BUCKET_NAMES, weights_matrix[best_idx]))

print("=" * 55)
print("MAX SHARPE PORTFOLIO — 6-Bucket Dual-Strait")
print("=" * 55)
print(f"  Annualised Return : {port_returns[best_idx]:.2%}")
print(f"  Annualised Vol    : {port_vols[best_idx]:.2%}")
print(f"  Sharpe Ratio      : {port_sharpes[best_idx]:.3f}")
print()
print("Bucket weights:")
for bucket, w in optimal_bucket_weights.items():
    print(f"  {bucket:<15} {w:.1%}")

print()
print("MIN VOL PORTFOLIO (for comparison)")
print("=" * 55)
print(f"  Annualised Return : {port_returns[minvol_idx]:.2%}")
print(f"  Annualised Vol    : {port_vols[minvol_idx]:.2%}")
print(f"  Sharpe Ratio      : {port_sharpes[minvol_idx]:.3f}")
print()
print("Bucket weights:")
for bucket, w in zip(BUCKET_NAMES, weights_matrix[minvol_idx]):
    print(f"  {bucket:<15} {w:.1%}")

## 7. Final Per-Stock Weights for QuantConnect

In [ ]:
final_weights = {}
for bucket_name, stocks in BUCKETS.items():
    bucket_w = optimal_bucket_weights[bucket_name]
    w_arr = np.array(list(stocks.values()))
    w_arr = w_arr / w_arr.sum()
    for ticker, within_w in zip(stocks.keys(), w_arr):
        final_weights[ticker] = bucket_w * within_w

total = sum(final_weights.values())
assert abs(total - 1.0) < 1e-9, f"Weights don't sum to 1: {total}"

print("Final per-stock weights (copy into QuantConnect):")
print()
print("HOLDINGS = {")
for bucket_name, stocks in BUCKETS.items():
    print(f"    # --- {bucket_name} ({optimal_bucket_weights[bucket_name]:.1%} of portfolio) ---")
    for ticker in stocks:
        print(f"    '{ticker}': {final_weights[ticker]:.6f},")
print("}")
print()
print(f"Total weight: {total:.8f}")

## 8. Bucket Weight Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#00BCD4']
bucket_to_color = dict(zip(BUCKET_NAMES, colors))

bucket_labels = [f"{k}\n{v:.1%}" for k, v in optimal_bucket_weights.items()]
axes[0].pie(
    list(optimal_bucket_weights.values()),
    labels=bucket_labels, colors=colors, autopct='',
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Dual-Strait Optimal Bucket Allocation (Max Sharpe)', fontsize=12)

tickers_sorted = sorted(final_weights, key=final_weights.get, reverse=True)
vals = [final_weights[t] for t in tickers_sorted]
ticker_to_bucket = {t: b for b, stocks in BUCKETS.items() for t in stocks}
bar_colors = [bucket_to_color[ticker_to_bucket[t]] for t in tickers_sorted]

axes[1].bar(tickers_sorted, [v * 100 for v in vals], color=bar_colors, edgecolor='white')
axes[1].set_ylabel('Portfolio Weight (%)', fontsize=11)
axes[1].set_title('Per-Stock Allocation — Dual-Strait Portfolio', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=bucket_to_color[b], label=b) for b in BUCKET_NAMES]
axes[1].legend(handles=legend_elements, fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('dual_strait_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dual_strait_weights.png")

## 9. Hormuz vs Dual-Strait: Side-by-Side Bucket Weight Comparison

Reference Hormuz weights (from the working backtest):
- Aluminum 12.6% | LNG/Gas 37.1% | Fertilizer 10.3% | Defense 21.1% | Tankers 18.9%

In [ ]:
hormuz_weights = {
    'Aluminum':   0.126,
    'LNG_Gas':    0.371,
    'Fertilizer': 0.103,
    'Defense':    0.211,
    'Tankers':    0.189,
    'Container':  0.000,
}

comparison_df = pd.DataFrame({
    'Hormuz-only': hormuz_weights,
    'Dual-Strait': optimal_bucket_weights,
})
comparison_df['Change (pp)'] = (comparison_df['Dual-Strait'] - comparison_df['Hormuz-only']) * 100

print("Bucket weight comparison:")
print(comparison_df.applymap(lambda x: f"{x:.1%}" if abs(x) < 10 else f"{x:+.1f}pp").to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(BUCKET_NAMES))
w = 0.35
ax.bar(x - w/2, [hormuz_weights[b] * 100 for b in BUCKET_NAMES], w,
       label='Hormuz-only', color='#78909C', edgecolor='white')
ax.bar(x + w/2, [optimal_bucket_weights[b] * 100 for b in BUCKET_NAMES], w,
       label='Dual-Strait', color='#1565C0', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(BUCKET_NAMES, rotation=20)
ax.set_ylabel('Bucket Weight (%)')
ax.set_title('Hormuz vs Dual-Strait: Bucket Weight Shift', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('dual_strait_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dual_strait_comparison.png")

---
## 10. Simulated Pre-Closure Run-Up: What Would the Dual-Strait Portfolio Have Made?

The Hormuz portfolio earned most of its gains *before* the Feb 28, 2026 official closure — the market priced in the disruption in advance. This section backtests the dual-strait portfolio from Jan 2025 to present and decomposes return into:
- **Pre-Hormuz-closure** (Jan 2025 – Feb 27, 2026): the period the Hormuz run-up happened
- **Post-Hormuz-closure** (Feb 28, 2026 – present): performance after the strait closed

If BaM tensions escalate, we expect a similar pre-closure run-up to emerge in the BaM-sensitive positions (Tankers, LNG, Container). This backtest shows the baseline performance before that additional catalyst.

In [ ]:
BACKTEST_START  = '2025-01-01'
HORMUZ_CLOSURE  = '2026-02-28'
TODAY           = pd.Timestamp.today().strftime('%Y-%m-%d')

raw_bt = yf.download(ALL_TICKERS, start=BACKTEST_START, end=TODAY, auto_adjust=True, progress=False)['Close']
# Also pull SPY and Hormuz portfolio holdings for comparison
hormuz_tickers = ['AA', 'CENX', 'RIO', 'LNG', 'EQT', 'AR', 'GLNG', 'CF', 'MOS', 'IPI',
                  'LMT', 'RTX', 'NOC', 'FRO', 'STNG', 'INSW']
spy_raw = yf.download(['SPY'] + hormuz_tickers, start=BACKTEST_START, end=TODAY,
                      auto_adjust=True, progress=False)['Close']

ret_bt = raw_bt.pct_change().ffill().dropna()

print(f"Backtest data: {ret_bt.shape[0]} trading days")
print(f"Period: {ret_bt.index[0].date()} → {ret_bt.index[-1].date()}")

In [ ]:
# Build dual-strait bucket return series over the full backtest period
bucket_ret_bt = pd.DataFrame(index=ret_bt.index)
for bucket_name, weights in BUCKETS.items():
    tickers = list(weights.keys())
    w = np.array(list(weights.values())); w /= w.sum()
    bucket_ret_bt[bucket_name] = (ret_bt[tickers] * w).sum(axis=1)

opt_bw = np.array([optimal_bucket_weights[b] for b in BUCKET_NAMES])
dual_daily = bucket_ret_bt.values @ opt_bw
dual_cumret = pd.Series((1 + dual_daily).cumprod(), index=ret_bt.index)

# Hormuz portfolio (fixed weights from the working backtest)
hormuz_final_weights = {
    'AA': 0.063000, 'CENX': 0.031500, 'RIO': 0.031500,
    'LNG': 0.092750, 'EQT': 0.092750, 'AR': 0.092750, 'GLNG': 0.092750,
    'CF': 0.034333, 'MOS': 0.034333, 'IPI': 0.034333,
    'LMT': 0.126600, 'RTX': 0.063300, 'NOC': 0.021100,
    'FRO': 0.063000, 'STNG': 0.063000, 'INSW': 0.063000,
}
hormuz_ticks = list(hormuz_final_weights.keys())
ret_h = spy_raw[hormuz_ticks].pct_change().ffill().dropna()
ret_h = ret_h.reindex(ret_bt.index).ffill().dropna()

hormuz_daily = (ret_h * pd.Series(hormuz_final_weights)).sum(axis=1)
hormuz_cumret = (1 + hormuz_daily).cumprod()

# SPY
spy_ret = spy_raw['SPY'].pct_change().dropna().reindex(ret_bt.index).ffill().dropna()
spy_cumret = (1 + spy_ret).cumprod()

# Align all to same index
common_idx = dual_cumret.index.intersection(hormuz_cumret.index).intersection(spy_cumret.index)
dual_cumret    = dual_cumret.reindex(common_idx)
hormuz_cumret  = hormuz_cumret.reindex(common_idx)
spy_cumret     = spy_cumret.reindex(common_idx)
bucket_cumret  = (1 + bucket_ret_bt).cumprod().reindex(common_idx)

closure_line = pd.Timestamp(HORMUZ_CLOSURE)
pre_idx  = common_idx[common_idx < closure_line]
post_idx = common_idx[common_idx >= closure_line]

def period_return(series, idx):
    if len(idx) < 2:
        return 0.0
    return series.loc[idx[-1]] / series.loc[idx[0]] - 1

print("Period decomposition:")
print(f"{'':30} {'Pre-closure':>15} {'Post-closure':>15} {'Full period':>12}")
print(f"{'':30} {'(Jan25–Feb26)':>15} {'(Mar26–now)':>15} {'(total)':>12}")
print("-" * 75)
for name, series in [('Dual-Strait', dual_cumret), ('Hormuz-only', hormuz_cumret), ('SPY', spy_cumret)]:
    pre  = period_return(series, pre_idx)
    post = period_return(series, post_idx)
    full = series.iloc[-1] - 1
    print(f"  {name:<28} {pre:>+14.1%} {post:>+15.1%} {full:>+11.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(dual_cumret.index,   (dual_cumret   - 1) * 100, color='#1565C0', linewidth=2.5, label='Dual-Strait')
ax.plot(hormuz_cumret.index, (hormuz_cumret - 1) * 100, color='#E65100', linewidth=2.0,
        linestyle='--', label='Hormuz-only (working backtest)')
ax.plot(spy_cumret.index,    (spy_cumret    - 1) * 100, color='gray',    linewidth=1.5,
        linestyle='-.', label='SPY (benchmark)')

if closure_line in common_idx or (common_idx.min() < closure_line < common_idx.max()):
    ax.axvline(closure_line, color='red', linewidth=1.2, linestyle=':', alpha=0.8,
               label='Hormuz closure (Feb 28, 2026)')

ax.axhline(0, color='lightgray', linewidth=0.8, linestyle=':')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.0f}%'))
ax.set_title('Dual-Strait vs Hormuz-only vs SPY\n(Jan 2025 → present)', fontsize=11)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.legend(fontsize=9)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

ax2 = axes[1]
colors6 = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#00BCD4']
bcolor = dict(zip(BUCKET_NAMES, colors6))
for bucket, color in bcolor.items():
    ax2.plot(bucket_cumret.index, (bucket_cumret[bucket] - 1) * 100,
             label=bucket, color=color, linewidth=1.8, alpha=0.85)
ax2.plot(dual_cumret.index, (dual_cumret - 1) * 100,
         color='black', linewidth=2.5, linestyle='--', label='Portfolio (dual-strait)')
if closure_line in common_idx or (common_idx.min() < closure_line < common_idx.max()):
    ax2.axvline(closure_line, color='red', linewidth=1.2, linestyle=':', alpha=0.8)
ax2.axhline(0, color='lightgray', linewidth=0.8, linestyle=':')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.0f}%'))
ax2.set_title('Dual-Strait: Bucket Performance\n(Jan 2025 → present)', fontsize=11)
ax2.set_xlabel('Date')
ax2.set_ylabel('Cumulative Return')
ax2.legend(fontsize=8)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Dual-Strait Portfolio — Backtest Performance (out-of-sample)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('dual_strait_backtest.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dual_strait_backtest.png")